# Лабораторная работа 6. Гибридная рекомендательная система SVD → ItemKNN → Content-Based

Цель работы — построить гибридную рекомендательную систему для видеоигр на основе трёх последовательных этапов: SVD → ItemKNN → Content-Based.

Для оценки применяется отдельное разделение взаимодействий на обучающую, валидационную и тестовую выборки. Для каждого пользователя история делится примерно в пропорции 80/10/10; если в validation или test случайно не попадает положительно оценённая игра, одна такая игра переносится из train в соответствующую часть. Поэтому при оценке у пользователя есть релевантные объекты и при этом они не используются в обучении.

SVD и ItemKNN обучаются только на train. Размеры кандидатных пулов выбираются по validation. Test используется один раз — для окончательного сравнения моделей. Каталог ограничивается наиболее частыми играми в train, что позволяет работать с большим датасетом без использования тестовых взаимодействий для отбора каталога.

# Гибридная рекомендательная система
## Работу выполнял студент *Простолупов Роман* группы *ИКНТ-ИИ-2023*

Был использован датасет *"Metacritic's Best Games and Reviews - 2025"*, в соответствии с поставленной задачей.

В данной работе простроена рекомендательная система на основе трёх этапов: **SVD -> ItemKNN -> Content Based(TF-IDF).**

Каждый из этапов передаёт следующему список выявленных кандидатов, что приводит к последовательному улучшению результатов:
* SVD использует скрытые закономерности в пользовательских оценках и выводит большой набор кандидатов.
* ItemKNN опирается на похожесть этих игр и выводит меньший набор кандидатов.
* Content-based подход дополнительно учитывает содержание игр и персональный контентный профиль пользователя.

In [68]:
import pandas as pd
import numpy as np
import time
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors

## 1. Загрузка данных

Датасет состоит из двух файлов:
- games.csv — информация об играх;
- games_reviews.csv — пользовательские оценки.

Загружаются оба набора данных, отсеиваются пропуски, проверяется их размер.

In [3]:
GAMES_PATH="games/games.csv"
REVIEWS_PATH="games/games_reviews.csv"

games=pd.read_csv(GAMES_PATH)
reviews=pd.read_csv(REVIEWS_PATH)

reviews["score"]=pd.to_numeric(reviews["score"],errors="coerce")
games=games.drop_duplicates("id").copy()
reviews=reviews.dropna(subset=["author","id","score"])
reviews=reviews[reviews["id"].isin(games["id"])].copy()
reviews=reviews.groupby(["author","id"],as_index=False)["score"].mean()

print("Games:",games.shape)
print("Unique user-game interactions:",reviews.shape)
print("\nGames columns:",games.columns.tolist())
print("\nReviews columns:",reviews.columns.tolist())

Games: (13442, 16)
Unique user-game interactions: (1088972, 3)

Games columns: ['id', 'title', 'releaseDate', 'rating', 'genres', 'description', 'platforms', 'metascore', 'metascore_count', 'metascore_sentiment', 'userscore', 'userscore_count', 'userscore_sentiment', 'platform_metascores', 'developer', 'publisher']

Reviews columns: ['author', 'id', 'score']


In [4]:
display(games.head())
display(reviews.head())

,id,title,releaseDate,rating,genres,description,platforms,metascore,metascore_count,metascore_sentiment,userscore,userscore_count,userscore_sentiment,platform_metascores,developer,publisher
0,1300001290,The Legend of Zelda: Ocarina of Time,1998-11-23,E,Open-World Action,"As a young boy, Link is tricked by Ganondorf, ...",Nintendo 64,99.0,22.0,Universal acclaim,91.0,10611,Universal acclaim,99,Nintendo,"Nintendo,Gradiente"
1,1300001928,SoulCalibur,1999-09-08,T,3D Fighting,"[Xbox Live Arcade] Soulcalibur, the highest M...","Dreamcast,iOS (iPhone/iPad),Xbox 360",98.0,24.0,Universal acclaim,78.0,605,Generally favorable,"98,73,79",Namco,Namco
2,1300027043,Grand Theft Auto IV,2008-04-29,M,Open-World Action,[Metacritic's 2008 PS3 Game of the Year; Also ...,"PlayStation 3,Xbox 360,PC",98.0,86.0,Universal acclaim,83.0,4781,Generally favorable,"98,98,90",Rockstar North,"Rockstar Games,Capcom"
3,1300019237,Super Mario Galaxy,2007-11-12,E,3D Platformer,[Metacritic's 2007 Wii Game of the Year] The u...,Wii,97.0,73.0,Universal acclaim,91.0,4181,Universal acclaim,97,Nintendo,Nintendo
4,1300044129,Super Mario Galaxy 2,2010-05-23,E,3D Platformer,"Super Mario Galaxy 2, the sequel to the galaxy...",Wii,97.0,87.0,Universal acclaim,90.0,2933,Universal acclaim,97,Nintendo EAD Tokyo,"Nintendo,iQue"


,author,id,score
0,--,1300499254,100.0
1,-DeathGun-,1300522193,20.0
2,-Gabriele-,1300461445,80.0
3,-Gabriele-,1300523817,90.0
4,-JustRamZero-,1300000205,80.0


В таблице games каждая строка соответствует одной игре.

Для Content-Based этапа используются следующие признаки: ``title, genres, description, platforms, developer, publisher.``

В таблице reviews хранятся пользовательские взаимодействия. Для построения коллаборативной части используются ``author, id и score.``

## 2. Параметры эксперимента и выбор пользователей

Полный датасет слишком велик для многократного обучения и оценки всех пользователей, поэтому далее выбирается ограниченная случайная подвыборка пользователей.
Затем взаимодействия каждого отдельного пользователя с играми будут разделены на обучающую и тестовые выборки, так как в рекомендательной системе важно оставить часть истории каждого пользователя в train, чтобы затем оценивать рекомендации именно для знакомых системе пользователей.

После разделения каталог игр будет ограничен по частоте взаимодействий в train. Таким образом, test-взаимодействия не влияют на выбор игр, по которым обучаются модели.

In [64]:
SEED=42
MIN_RATING_LIKE=70
MIN_USER_INTERACTIONS=5

MAX_USERS=5000
MAX_GAMES=20000
MAX_EVAL_USERS=2000

MAX_PROFILE_ITEMS=50
TFIDF_MAX_FEATURES=40000
NGRAM_RANGE=(1,4)

SVD_COMPONENTS=50
SVD_CANDIDATES=500

ITEM_KNN_NEIGHBORS=100
KNN_POOL=50

PROGRESS_EVERY=100 # Для отслеживания продолжительности вычислений

rng=np.random.default_rng(SEED)

### Выбор пользователей до разделения

В эксперимент включаются пользователи с достаточным количеством взаимодействий и минимум тремя положительно оценёнными играми. Это нужно, чтобы после разбиения у пользователя оставались положительные примеры для обучения, validation и test.

На этом шаге список игр ещё не ограничивается; сначала разделяется история, а каталог формируется позже только по train.

In [6]:
user_counts=reviews.groupby("author").size()
positive_counts=reviews.loc[reviews["score"]>=MIN_RATING_LIKE].groupby("author").size()
eligible_users=user_counts[user_counts>=MIN_USER_INTERACTIONS].index.intersection(positive_counts[positive_counts>=3].index).to_numpy()

n_users=min(MAX_USERS,len(eligible_users))
sampled_users=rng.choice(eligible_users,size=n_users,replace=False)
sample_reviews=reviews[reviews["author"].isin(sampled_users)].copy().reset_index(drop=True)

print("Пользователей в подвыборке:",sample_reviews["author"].nunique())
print("Игр до ограничения каталога:",sample_reviews["id"].nunique())
print("Уникальных взаимодействий:",len(sample_reviews))

Пользователей в подвыборке: 5000
Игр до ограничения каталога: 7303
Уникальных взаимодействий: 70260


## 3. Разделение на train, validation и test

История оценок каждого пользователя разбивается отдельно: ориентировочно 80% взаимодействий отправляются в train, 10% — в validation и 10% — в test.

Выборки Validation и Test формируются таким образом, чтобы содержать не менее одной положительно оценённой игры. Для достижения этого ценки игр пользователями могут переноситься из train в val. или test при необходимости. Для каждого отдельного пользователя все разделы остаются непересекающимися.

In [7]:
split_rng=np.random.default_rng(SEED)
train_parts=[]; val_parts=[]; test_parts=[]

def move_positive(target, donor):
    if (target["score"]>=MIN_RATING_LIKE).any(): return target,donor
    pos=donor[donor["score"]>=MIN_RATING_LIKE]
    if pos.empty or target.empty: return target,donor
    pos_idx=pos.index[0]; replace_idx=target.index[0]
    pos_row=donor.loc[[pos_idx]]; replace_row=target.loc[[replace_idx]]
    target=pd.concat([target.drop(index=replace_idx),pos_row])
    donor=pd.concat([donor.drop(index=pos_idx),replace_row])
    return target,donor

for user_id,group in sample_reviews.groupby("author",sort=False):
    group=group.iloc[split_rng.permutation(len(group))]
    n_val=max(1,int(np.floor(len(group)*0.1)))
    n_test=max(1,int(np.floor(len(group)*0.1)))
    if n_val+n_test>=len(group): n_val,n_test=1,1
    val=group.iloc[:n_val].copy()
    test=group.iloc[n_val:n_val+n_test].copy()
    train=group.iloc[n_val+n_test:].copy()
    val,train=move_positive(val,train)
    test,train=move_positive(test,train)
    train_parts.append(train); val_parts.append(val); test_parts.append(test)

train_reviews=pd.concat(train_parts,ignore_index=True)
val_reviews=pd.concat(val_parts,ignore_index=True)
test_reviews=pd.concat(test_parts,ignore_index=True)

print("Разделение до ограничения каталога:")
for name,df in [("Train",train_reviews),("Validation",val_reviews),("Test",test_reviews)]:
    print(f"{name}: {len(df):,} ({len(df)/len(sample_reviews):.1%})")

# Каталог игр выбирается только по обучающим взаимодействиям.
train_game_counts=train_reviews.groupby("id").size().sort_values(ascending=False)
sample_game_ids=train_game_counts.head(MAX_GAMES).index

games=games[games["id"].isin(sample_game_ids)].copy().reset_index(drop=True)
train_reviews=train_reviews[train_reviews["id"].isin(sample_game_ids)].copy()
val_reviews=val_reviews[val_reviews["id"].isin(sample_game_ids)].copy()
test_reviews=test_reviews[test_reviews["id"].isin(sample_game_ids)].copy()
sample_reviews=sample_reviews[sample_reviews["id"].isin(sample_game_ids)].copy()

# Оставляем пользователей, у которых после ограничения каталога есть train-профиль,
# а также хотя бы одна положительная игра в validation и test.
train_pos_users=set(train_reviews.loc[train_reviews["score"]>=MIN_RATING_LIKE,"author"])
val_pos_users=set(val_reviews.loc[val_reviews["score"]>=MIN_RATING_LIKE,"author"])
test_pos_users=set(test_reviews.loc[test_reviews["score"]>=MIN_RATING_LIKE,"author"])
valid_users=set(sampled_users)&train_pos_users&val_pos_users&test_pos_users

eval_users=np.array([u for u in sampled_users if u in valid_users])
train_reviews=train_reviews[train_reviews["author"].isin(valid_users)].copy().reset_index(drop=True)
val_reviews=val_reviews[val_reviews["author"].isin(valid_users)].copy().reset_index(drop=True)
test_reviews=test_reviews[test_reviews["author"].isin(valid_users)].copy().reset_index(drop=True)
sample_reviews=sample_reviews[sample_reviews["author"].isin(valid_users)].copy().reset_index(drop=True)

train_keys=set(zip(train_reviews["author"],train_reviews["id"]))
val_keys=set(zip(val_reviews["author"],val_reviews["id"]))
test_keys=set(zip(test_reviews["author"],test_reviews["id"]))
assert train_keys.isdisjoint(val_keys) and train_keys.isdisjoint(test_keys) and val_keys.isdisjoint(test_keys)
assert len(eval_users)>0, "После ограничения каталога не осталось пользователей для оценки; увеличьте MAX_GAMES или MAX_USERS."

print("\nПосле ограничения каталога, выбранного только по train:")
print("Пользователей для оценки:",len(eval_users))
print("Игр в каталоге:",games["id"].nunique())
print("Train:",len(train_reviews),"| Validation:",len(val_reviews),"| Test:",len(test_reviews))
print("Пересечения пар user-game: train/val =",len(train_keys&val_keys),", train/test =",len(train_keys&test_keys),", val/test =",len(val_keys&test_keys))

Разделение до ограничения каталога:
Train: 55,538 (79.0%)
Validation: 7,361 (10.5%)
Test: 7,361 (10.5%)

После ограничения каталога, выбранного только по train:
Пользователей для оценки: 4742
Игр в каталоге: 6792
Train: 53333 | Validation: 6921 | Test: 6942
Пересечения пар user-game: train/val = 0 , train/test = 0 , val/test = 0


## 4. Построение пользовательско-игровой матрицы

Матрица user-item строится исключительно по train. Строки соответствуют пользователям, столбцы — играм из каталога, значение ячейки — оценка пользователя, масштабированная в диапазон от 0 до 1.

Также для каждого пользователя формируются списки знакомых и понравившихся игр, чтобы упростить исключение уже оценённых игр из рекомендаций.

In [8]:
user_ids=train_reviews["author"].drop_duplicates().to_numpy()
item_ids=games["id"].to_numpy()
user_to_idx={u:i for i,u in enumerate(user_ids)}
item_to_idx={x:i for i,x in enumerate(item_ids)}

train_reviews=train_reviews.copy()
train_reviews["u_idx"]=train_reviews["author"].map(user_to_idx)
train_reviews["i_idx"]=train_reviews["id"].map(item_to_idx)
score_scale=100 if train_reviews["score"].max()>1 else 1

train_matrix=csr_matrix(
    (train_reviews["score"].to_numpy(dtype=np.float32)/score_scale,
     (train_reviews["u_idx"],train_reviews["i_idx"])),
    shape=(len(user_ids),len(item_ids))
)

train_liked=train_reviews[train_reviews["score"]>=MIN_RATING_LIKE]
liked_by_user={u:grp["i_idx"].to_numpy() for u,grp in train_liked.groupby("u_idx")}
seen_by_user={u:grp["i_idx"].to_numpy() for u,grp in train_reviews.groupby("u_idx")}

val_items_by_user={user_to_idx[u]:set(item_to_idx[i] for i in grp["id"] if i in item_to_idx)
                   for u,grp in val_reviews[val_reviews["score"]>=MIN_RATING_LIKE].groupby("author") if u in user_to_idx}
test_items_by_user={user_to_idx[u]:set(item_to_idx[i] for i in grp["id"] if i in item_to_idx)
                    for u,grp in test_reviews[test_reviews["score"]>=MIN_RATING_LIKE].groupby("author") if u in user_to_idx}
eval_user_idx=np.array([user_to_idx[u] for u in eval_users if u in user_to_idx and user_to_idx[u] in val_items_by_user and user_to_idx[u] in test_items_by_user])

print("Матрица user-item:",train_matrix.shape)
print("Ненулевых значений:",train_matrix.nnz)
print("Плотность:",round(train_matrix.nnz/(train_matrix.shape[0]*train_matrix.shape[1]),6))
print("Положительных validation-пользователей:",len(val_items_by_user))
print("Положительных test-пользователей:",len(test_items_by_user))
print("Пользователей для оценки:",len(eval_user_idx))

Матрица user-item: (4742, 6792)
Ненулевых значений: 53333
Плотность: 0.001656
Положительных validation-пользователей: 4742
Положительных test-пользователей: 4742
Пользователей для оценки: 4742


In [73]:
print("Пользователей:", train_matrix.shape[0])
print("Игр:", train_matrix.shape[1])
print("Взаимодействий:", train_matrix.nnz)
print("Плотность:",
      train_matrix.nnz / (train_matrix.shape[0] * train_matrix.shape[1]))
print("Среднее число оценок у пользователя:", train_matrix.nnz / train_matrix.shape[0])
print("Среднее число оценок у игры:", train_matrix.nnz / train_matrix.shape[1])

Пользователей: 4742
Игр: 6792
Взаимодействий: 53333
Плотность: 0.0016559102206232653
Среднее число оценок у пользователя: 11.246942218473219
Среднее число оценок у игры: 7.852326266195524


## 5. Первый этап гибридного подхода - SVD

На первом этапе сингулярное разложение используется непосредственно для матрицы *пользователь–игра*. Матрица раскладывается на скрытые факторы пользователей и игр. Для каждого пользователя рассчитывается оценка всех игр. После этого выбирается широкий пул SVD_CANDIDATES кандидатов.

На этом этапе нужно оставить достаточно много кандидатов, так как следующие этапы будут фильтровать этот набор.

In [60]:
t0=time.time()

svd_model=TruncatedSVD(n_components=SVD_COMPONENTS,algorithm="randomized",n_iter=5,random_state=SEED)
user_factors=svd_model.fit_transform(train_matrix)

print("User factors:",user_factors.shape)
print("Item factors:",svd_model.components_.shape)
print("Explained variance:",round(svd_model.explained_variance_ratio_.sum(),4))
print("SVD time:",round(time.time()-t0,2),"sec")

User factors: (4742, 50)
Item factors: (50, 6792)
Explained variance: 0.257
SVD time: 0.06 sec


## 6.  Второй этап - ItemKNN

Этот подход ищет игры, которые имеют похожие пользовательские оценки.

Для оптимизации дальнейших вычислительных процессов в матрицу сходств для каждой из игр сохраняются данные о дистанциях (ITEM_KNN_NEIGHBORS) ближайших игр-соседей.

После получения SVD-кандидатов ItemKNN используется для их дополнительного ранжирования.

In [10]:
t0=time.time()

item_knn=NearestNeighbors(
    n_neighbors=ITEM_KNN_NEIGHBORS+1,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

item_knn.fit(train_matrix.T)
distances,neighbors=item_knn.kneighbors(train_matrix.T,return_distance=True)

knn_rows=np.repeat(np.arange(len(item_ids)),ITEM_KNN_NEIGHBORS)
knn_cols=neighbors[:,1:].ravel()
knn_data=(1-distances[:,1:]).ravel().astype(np.float32)

mask=knn_data>0
item_similarity=csr_matrix(
    (knn_data[mask],(knn_rows[mask],knn_cols[mask])),
    shape=(len(item_ids),len(item_ids))
)

print("Item similarity:",item_similarity.shape)
print("Ненулевых связей:",item_similarity.nnz)
print("ItemKNN time:",round(time.time()-t0,2),"sec")

Item similarity: (6792, 6792)
Ненулевых связей: 442329
ItemKNN time: 0.51 sec


## 7. Content-Based представление игр

Текстовое описание игры создаётся посредством объединения следующих данных (как говорилось ранее): ``title, genres, description, platforms, developer и publisher.``

После этого используется метод TF-IDF, который создаёт для каждого сочетания слов _показатель частотности в пределах всей выборки_.

Content-Based будет сравнивать профиль пользователя с небольшим пулом игр, уже отфильтрованным предыдущими этапами SVD и ItemKNN.

In [22]:
text_cols=["title","genres","description","platforms","developer","publisher"]

for c in text_cols:
    games[c]=games[c].fillna("").astype(str)

games["content_text"]=games[text_cols].agg(" ".join,axis=1).str.lower().str.replace(r"\s+"," ",regex=True).str.strip()

t0=time.time()

vectorizer=TfidfVectorizer(
    stop_words="english",
    ngram_range=NGRAM_RANGE,
    max_features=TFIDF_MAX_FEATURES,
    min_df=2,
    sublinear_tf=True
)

tfidf_matrix=vectorizer.fit_transform(games["content_text"])
tfidf_norm=normalize(tfidf_matrix,copy=False)

print("TF-IDF shape:",tfidf_matrix.shape)
print("Features:",len(vectorizer.get_feature_names_out()))
print("TF-IDF time:",round(time.time()-t0,2),"sec")

TF-IDF shape: (6792, 40000)
Features: 40000
TF-IDF time: 3.91 sec


## 8. Вспомогательные функции

Далее реализуются функции для быстрого получения Top-K кандидатов из ряда оценок релевантности и списка оценённых игр.

Также заранее строится список популярных игр для резервного заполнения, если некоторый алгоритм сформировал слишком мало кандидатов.

In [23]:
popular_items=np.argsort(np.asarray(train_matrix.sum(axis=0)).ravel())[::-1]

def topk_dense(scores,k,seen):
    scores=scores.copy()
    seen=np.asarray(list(seen),dtype=int)
    if len(seen): scores[seen]=-np.inf
    k=min(k,len(scores)-len(seen))
    if k<=0: return np.array([],dtype=int)
    idx=np.argpartition(scores,-k)[-k:]
    return idx[np.argsort(scores[idx])[::-1]]

def topk_sparse(row,k,seen):
    idx=row.indices
    vals=row.data
    if len(seen):
        mask=~np.isin(idx,np.asarray(list(seen),dtype=int))
        idx,vals=idx[mask],vals[mask]
    if len(idx):
        order=np.argsort(vals)[::-1]
        result=list(idx[order[:k]])
    else:
        result=[]
    used=set(result)|set(seen)
    for x in popular_items:
        if x not in used:
            result.append(x)
            used.add(x)
        if len(result)>=k: break
    return np.asarray(result[:k],dtype=int)

## 9. Гибрид SVD → ItemKNN → Content-Based

Для формирования итогового списка рекомендаций выполняются три последовательных этапа:

1. SVD формирует широкий пул кандидатов.

2. ItemKNN получает только этот пул и выбирает наиболее близкие по пользовательскому поведению игры.

3. Content-Based рассматривает уже небольшой пул после ItemKNN и выполняет окончательное ранжирование по сходству содержания с профилем пользователя.

Функция возвращает (top_n) рекомендаций отдельно по каждому из этапов, что позволяет сравнить эффективность каждого из них

In [24]:
def recommend_pipeline(uidx,top_n=10):
    seen=set(seen_by_user.get(uidx,[]))
    liked=liked_by_user.get(uidx,np.array([],dtype=int))
    if len(liked)==0: return {},{},None

    svd_scores=user_factors[uidx] @ svd_model.components_
    svd_candidates=topk_dense(svd_scores,SVD_CANDIDATES,seen)
    knn_row=train_matrix[uidx].dot(item_similarity)
    svd_knn_scores=knn_row[:,svd_candidates].toarray().ravel()
    order=np.argsort(-svd_knn_scores,kind="stable")
    knn_pool=svd_candidates[order[:KNN_POOL]]

    profile_idx=liked[-MAX_PROFILE_ITEMS:]
    profile=normalize(csr_matrix(np.asarray(tfidf_norm[profile_idx].mean(axis=0))))
    cb_scores=(profile @ tfidf_norm[knn_pool].T).toarray().ravel()
    cb_order=np.argsort(cb_scores)[::-1]
    hybrid=knn_pool[cb_order[:top_n]]

    svd_top=svd_candidates[:top_n]
    itemknn_top=topk_sparse(knn_row,top_n,seen)
    svd_knn_top=knn_pool[:top_n]
    recs={"SVD":svd_top,"ItemKNN":itemknn_top,"SVD→ItemKNN":svd_knn_top,"Hybrid":hybrid}
    scores={"SVD":svd_scores[svd_top],"ItemKNN":knn_row[:,itemknn_top].toarray().ravel(),
            "SVD→ItemKNN":svd_knn_scores[order[:top_n]],"Hybrid":cb_scores[cb_order[:top_n]]}
    return recs,scores,knn_pool

## 10. Просмотр рекомендаций для одного пользователя

Здесь работа рекомендательной системы проверяется на одном пользователе.

Показаны рекомендации, сформированные разными этапами:

SVD,

ItemKNN,

SVD → ItemKNN,

SVD → ItemKNN → Content-Based.

Это позволяет визуально увидеть, как набор рекомендаций изменяется после прохождения каждого этапа.

In [25]:
uidx=int(eval_user_idx[4000])
user_id=user_ids[uidx]
print("User:",user_id)
train_game_ids=train_reviews.loc[train_reviews["author"]==user_id,"id"]
display(games[games["id"].isin(train_game_ids)][["id","title","genres"]].head(20))
recs,scores,pool=recommend_pipeline(uidx,10)
for name,idx in recs.items():
    print("\n",name)
    display(games.iloc[idx][["id","title","genres"]].assign(score=scores[name]))

User: Tomas95


,id,title,genres
36,1300004716,Grand Theft Auto: Vice City,Open-World Action
43,1300126883,The Last of Us,Linear Action Adventure
65,1300597505,Metaphor: ReFantazio,JRPG
121,1300571786,Resident Evil 4,Survival
430,1300553298,Alan Wake II,Survival
479,1300518981,It Takes Two,Linear Action Adventure
775,1300580577,Silent Hill 2,Survival
924,1300571578,Star Wars Jedi: Survivor,Open-World Action
1489,1300024446,Alan Wake,Survival
1616,1300553301,"Warhammer 40,000: Space Marine II",Tactical Third Person Shooter



 SVD


,id,title,genres,score
384,1300540299,Dead Space,Survival,0.334390
2027,1300499254,Stellar Blade,Action Adventure,0.314478
1647,1300538287,A Plague Tale: Requiem,Linear Action Adventure,0.284173
4709,1300479228,Atomic Heart,FPS,0.282097
2621,1300517212,Mafia: Definitive Edition,Open-World Action,0.263150
2143,1300538302,Marvel's Guardians of the Galaxy,Linear Action Adventure,0.213714
1783,1300468660,A Plague Tale: Innocence,Linear Action Adventure,0.211895
2308,1300536979,Lies of P,Action Adventure,0.209812
381,1300577900,Cyberpunk 2077: Phantom Liberty,Action RPG,0.208427
863,1300518496,Returnal,Third Person Shooter,0.206279



 ItemKNN


,id,title,genres,score
5395,1300533291,Immortals Fenyx Rising: Myths of the Eastern R...,Open-World Action,1.509014
4294,1300530874,Immortals Fenyx Rising: A New God,Open-World Action,1.509014
5448,1300506883,Little Orpheus,2D Platformer,1.431352
3510,1300591447,Beat Slayer,3D Beat-'Em-Up,1.431352
3650,1300595976,Metal Gear Solid: Master Collection Vol. 1,Compilation,1.431352
5127,1300543473,Destroy All Humans! 2 - Reprobed,Action Adventure,1.431352
3881,1300500948,Killsquad,Action RPG,1.431352
3395,1300486985,Prey: Mooncrash,Action RPG,1.401084
5998,1300522310,Suicide Squad: Kill The Justice League,Third Person Shooter,1.060238
5544,1300577212,Atlas Fallen,Action RPG,1.043794



 SVD→ItemKNN


,id,title,genres,score
4294,1300530874,Immortals Fenyx Rising: A New God,Open-World Action,1.509014
5395,1300533291,Immortals Fenyx Rising: Myths of the Eastern R...,Open-World Action,1.509014
3510,1300591447,Beat Slayer,3D Beat-'Em-Up,1.431352
5448,1300506883,Little Orpheus,2D Platformer,1.431352
3881,1300500948,Killsquad,Action RPG,1.431352
5127,1300543473,Destroy All Humans! 2 - Reprobed,Action Adventure,1.431352
3650,1300595976,Metal Gear Solid: Master Collection Vol. 1,Compilation,1.431352
3395,1300486985,Prey: Mooncrash,Action RPG,1.401084
5998,1300522310,Suicide Squad: Kill The Justice League,Third Person Shooter,1.060238
5544,1300577212,Atlas Fallen,Action RPG,1.043794



 Hybrid


,id,title,genres,score
9,1300110124,Grand Theft Auto V,Open-World Action,0.155084
4259,1300617416,Alan Wake II: The Lake House Expansion,Survival,0.142558
3271,1300486966,Dying Light 2 Stay Human,Survival,0.137229
2308,1300536979,Lies of P,Action Adventure,0.126072
443,1300572186,The Last of Us Part I,Linear Action Adventure,0.122351
384,1300540299,Dead Space,Survival,0.121124
4067,1300528630,Evil West,Third Person Shooter,0.114873
3155,1300486974,Sable,Open-World Action,0.107991
5998,1300522310,Suicide Squad: Kill The Justice League,Third Person Shooter,0.105309
3142,1300577747,Assassin's Creed Mirage,Open-World Action,0.103624


## 12. Метрики

Итоговые метрики вычисляются только по положительно оценённым играм в test. Test может содержать несколько релевантных игр для пользователя, поэтому Recall@10 показывает долю найденных тестовых релевантных игр, а HitRate@10 — был ли найден хотя бы один такой объект. NDCG@10 дополнительно учитывает позиции найденных объектов.

Validation используется отдельно для выбора SVD_CANDIDATES и KNN_POOL. После подбора параметров итоговая модель оценивается на test; результаты test не используются для изменения конфигурации.

In [26]:
def calc_metrics(recommended,relevant,k=10):
    relevant=set(relevant)
    recommended=list(recommended[:k])
    hits=[1 if x in relevant else 0 for x in recommended]
    precision=sum(hits)/k
    recall=sum(hits)/len(relevant) if relevant else 0
    dcg=sum(h/np.log2(i+2) for i,h in enumerate(hits))
    ideal=sum(1/np.log2(i+2) for i in range(min(len(relevant),k)))
    ndcg=dcg/ideal if ideal else 0
    hit=int(sum(hits)>0)
    return precision,recall,ndcg,hit

## 12. Настройка гиперпараметров и размеров кандидатного пула
Здесь исследуется влияние на результат параметров:
* Пулы кандидатов от SVD и ItemKNN;
* Количество компонентов разложения SVD;
* Количество соседей ItemKNN.

Этот этап задействует validation-выборку.

In [65]:
SVD_COMPONENTS_LIST=[10, 25, 50,100,200]
ITEM_KNN_NEIGHBORS_LIST=[50,100,200]
SVD_CANDIDATES_LIST=[100, 300, 500,1000,1500]
KNN_POOL_LIST=[25, 50,100,200]
TUNING_USERS=len(eval_user_idx)
tuning_users=eval_user_idx

svd_models={int(SVD_COMPONENTS):(svd_model,user_factors)}

for n_components in SVD_COMPONENTS_LIST:
    if n_components in svd_models: continue
    t0=time.time()
    print(f"Обучение SVD: SVD_COMPONENTS={n_components}...")
    model=TruncatedSVD(n_components=n_components,algorithm="randomized",n_iter=5,random_state=SEED)
    factors=model.fit_transform(train_matrix)
    svd_models[n_components]=(model,factors)
    print(f"Готово за {time.time()-t0:.1f} сек. Explained variance: {model.explained_variance_ratio_.sum():.4f}")

def build_item_similarity(n_neighbors):
    n_items=train_matrix.shape[1]
    actual_neighbors=min(int(n_neighbors),n_items-1)
    model=NearestNeighbors(n_neighbors=actual_neighbors+1,metric="cosine",algorithm="brute",n_jobs=-1)
    model.fit(train_matrix.T)
    distances,neighbors=model.kneighbors(train_matrix.T,return_distance=True)
    knn_rows=np.repeat(np.arange(n_items),neighbors.shape[1]-1)
    knn_cols=neighbors[:,1:].ravel()
    knn_data=(1-distances[:,1:]).ravel().astype(np.float32)
    mask=knn_data>0
    return csr_matrix((knn_data[mask],(knn_rows[mask],knn_cols[mask])),shape=(n_items,n_items))

item_similarity_by_neighbors={int(ITEM_KNN_NEIGHBORS):item_similarity}

for n_neighbors in ITEM_KNN_NEIGHBORS_LIST:
    if n_neighbors in item_similarity_by_neighbors: continue
    t0=time.time()
    print(f"Построение ItemKNN: ITEM_KNN_NEIGHBORS={n_neighbors}...")
    item_similarity_by_neighbors[n_neighbors]=build_item_similarity(n_neighbors)
    print(f"Готово за {time.time()-t0:.1f} сек. Связей: {item_similarity_by_neighbors[n_neighbors].nnz:,}")

tuning_cache={}

for uidx in tuning_users:
    liked=liked_by_user.get(uidx,np.array([],dtype=int))
    if len(liked)==0 or uidx not in val_items_by_user: continue
    profile_idx=liked[-MAX_PROFILE_ITEMS:]
    profile=normalize(csr_matrix(np.asarray(tfidf_norm[profile_idx].mean(axis=0))))
    tuning_cache[uidx]=(set(seen_by_user.get(uidx,[])),profile,set(val_items_by_user[uidx]))

tuning_rows=[]
total_steps=len(tuning_cache)*len(SVD_COMPONENTS_LIST)*len(ITEM_KNN_NEIGHBORS_LIST)
processed=0
t0=time.time()

for n_components in SVD_COMPONENTS_LIST:
    model,factors=svd_models[n_components]
    for n_neighbors in ITEM_KNN_NEIGHBORS_LIST:
        similarity=item_similarity_by_neighbors[n_neighbors]
        print(f"Проверка: SVD_COMPONENTS={n_components}, ITEM_KNN_NEIGHBORS={n_neighbors}")
        for uidx,(seen,profile,relevant) in tuning_cache.items():
            svd_scores=factors[uidx] @ model.components_
            max_cand=topk_dense(svd_scores,max(SVD_CANDIDATES_LIST),seen)
            knn_row=train_matrix[uidx].dot(similarity)
            knn_scores=knn_row[:,max_cand].toarray().ravel()
            cb_scores=(profile @ tfidf_norm[max_cand].T).toarray().ravel()

            for cand_n in SVD_CANDIDATES_LIST:
                cand=max_cand[:cand_n]
                knn_sub=knn_scores[:len(cand)]
                knn_order=np.argsort(-knn_sub,kind="stable")

                for pool_n in KNN_POOL_LIST:
                    pool_pos=knn_order[:pool_n]
                    cb_order=np.argsort(-cb_scores[pool_pos],kind="stable")
                    rec=cand[pool_pos[cb_order[:10]]]
                    p,r,nd,h=calc_metrics(rec,relevant,10)
                    tuning_rows.append([n_components,n_neighbors,cand_n,pool_n,p,r,nd,h])

            processed+=1
            if processed==1 or processed%PROGRESS_EVERY==0 or processed==total_steps:
                elapsed=time.time()-t0
                rate=processed/elapsed if elapsed else 0
                eta=(total_steps-processed)/rate if rate else 0
                print(f"Прогресс: {processed}/{total_steps} ({processed/total_steps*100:.1f}%) | ETA {eta/60:.1f} мин")

tuning_results=pd.DataFrame(tuning_rows,columns=["SVD_COMPONENTS","ITEM_KNN_NEIGHBORS","SVD_CANDIDATES","KNN_POOL","Precision@10","Recall@10","NDCG@10","HitRate@10"])
param_cols=["SVD_COMPONENTS","ITEM_KNN_NEIGHBORS","SVD_CANDIDATES","KNN_POOL"]
metric_cols=["Precision@10","Recall@10","NDCG@10","HitRate@10"]
tuning_summary=tuning_results.groupby(param_cols,as_index=False)[metric_cols].mean()

print("Лучшие конфигурации по NDCG@10:")
display(tuning_summary.sort_values("NDCG@10",ascending=False).head(20))

print("Средние метрики при разных SVD_COMPONENTS:")
display(tuning_summary.groupby("SVD_COMPONENTS")[metric_cols].mean().sort_values("NDCG@10",ascending=False))

print("Средние метрики при разных ITEM_KNN_NEIGHBORS:")
display(tuning_summary.groupby("ITEM_KNN_NEIGHBORS")[metric_cols].mean().sort_values("NDCG@10",ascending=False))

Обучение SVD: SVD_COMPONENTS=10...
Готово за 0.0 сек. Explained variance: 0.1000
Обучение SVD: SVD_COMPONENTS=25...
Готово за 0.0 сек. Explained variance: 0.1719
Обучение SVD: SVD_COMPONENTS=100...
Готово за 0.2 сек. Explained variance: 0.3740
Обучение SVD: SVD_COMPONENTS=200...
Готово за 0.4 сек. Explained variance: 0.5216
Построение ItemKNN: ITEM_KNN_NEIGHBORS=50...
Готово за 0.5 сек. Связей: 256,837
Построение ItemKNN: ITEM_KNN_NEIGHBORS=200...
Готово за 0.6 сек. Связей: 726,081
Проверка: SVD_COMPONENTS=10, ITEM_KNN_NEIGHBORS=50
Прогресс: 1/71130 (0.0%) | ETA 4.6 мин
Прогресс: 100/71130 (0.1%) | ETA 2.1 мин
Прогресс: 200/71130 (0.3%) | ETA 2.1 мин
Прогресс: 300/71130 (0.4%) | ETA 2.0 мин
Прогресс: 400/71130 (0.6%) | ETA 2.0 мин
Прогресс: 500/71130 (0.7%) | ETA 2.0 мин
Прогресс: 600/71130 (0.8%) | ETA 2.0 мин
Прогресс: 700/71130 (1.0%) | ETA 2.0 мин
Прогресс: 800/71130 (1.1%) | ETA 2.0 мин
Прогресс: 900/71130 (1.3%) | ETA 2.0 мин
Прогресс: 1000/71130 (1.4%) | ETA 2.0 мин
Прогресс: 11

,SVD_COMPONENTS,ITEM_KNN_NEIGHBORS,SVD_CANDIDATES,KNN_POOL,Precision@10,Recall@10,NDCG@10,HitRate@10
44,10,200,300,25,0.017819,0.143417,0.082125,0.167440
24,10,100,300,25,0.017841,0.142834,0.081493,0.167229
100,25,200,100,25,0.018473,0.144429,0.080228,0.172923
40,10,200,100,25,0.018199,0.143267,0.080223,0.169338
4,10,50,300,25,0.018030,0.141733,0.079924,0.167651
60,25,50,100,25,0.017925,0.140839,0.079419,0.167651
160,50,200,100,25,0.018051,0.141725,0.079184,0.168705
48,10,200,500,25,0.016723,0.137786,0.078778,0.159216
80,25,100,100,25,0.018283,0.142502,0.078562,0.170392
0,10,50,100,25,0.017714,0.139005,0.078536,0.164066


Средние метрики при разных SVD_COMPONENTS:


,Precision@10,Recall@10,NDCG@10,HitRate@10
SVD_COMPONENTS,,,,
10,0.015470,0.122894,0.070035,0.144956
25,0.015363,0.121855,0.069759,0.144134
50,0.015016,0.119415,0.068570,0.141329
100,0.014333,0.115621,0.066582,0.136124
200,0.013402,0.109964,0.063598,0.128596


Средние метрики при разных ITEM_KNN_NEIGHBORS:


,Precision@10,Recall@10,NDCG@10,HitRate@10
ITEM_KNN_NEIGHBORS,,,,
200,0.014853,0.119030,0.068480,0.140325
50,0.014720,0.117781,0.067469,0.139036
100,0.014577,0.117038,0.067178,0.137722


Поскольку в тестовой/валидационной выборке игр на каждого пользователя приходится довольно мало, а число рекомендуемых игр здесь равно десяти, что большинстве случаев сильно больше тестовой/валидационной выборки, метрики дают весьма низкий результат.

## 14. Выбор параметров гибридной модели

Для выбора конфигурации используется NDCG@10 на validation.

После выполнения предыдущей ячейки выбирается конфигурация с максимальным NDCG@10 на validation. До завершения финальной оценки выбранные параметры больше не изменяются.

In [66]:
best_config=tuning_summary.sort_values("NDCG@10",ascending=False).iloc[0]

SVD_COMPONENTS=int(best_config["SVD_COMPONENTS"])
ITEM_KNN_NEIGHBORS=int(best_config["ITEM_KNN_NEIGHBORS"])
SVD_CANDIDATES=int(best_config["SVD_CANDIDATES"])
KNN_POOL=int(best_config["KNN_POOL"])

svd_model,user_factors=svd_models[SVD_COMPONENTS]
item_similarity=item_similarity_by_neighbors[ITEM_KNN_NEIGHBORS]

print("Выбранные параметры:")
print("SVD_COMPONENTS =",SVD_COMPONENTS)
print("ITEM_KNN_NEIGHBORS =",ITEM_KNN_NEIGHBORS)
print("SVD_CANDIDATES =",SVD_CANDIDATES)
print("KNN_POOL =",KNN_POOL)
print("Explained variance =",round(svd_model.explained_variance_ratio_.sum(),4))

Выбранные параметры:
SVD_COMPONENTS = 10
ITEM_KNN_NEIGHBORS = 200
SVD_CANDIDATES = 300
KNN_POOL = 25
Explained variance = 0.1


## 15. Итоговая оценка на тестовой выборке

Test не использовался при обучении SVD и ItemKNN, построении пользовательских профилей или подборе размеров кандидатного пула. Теперь на нём сравниваются четыре варианта: SVD, ItemKNN, SVD → ItemKNN и SVD → ItemKNN → Content-Based.

Для каждого пользователя релевантными считаются игры из его test-части с оценкой не ниже MIN_RATING_LIKE.

In [67]:
results=[]
total=len(eval_user_idx)
t0=time.time()

for n,uidx in enumerate(eval_user_idx,1):
    relevant=test_items_by_user.get(uidx,set())
    if not relevant: continue
    recs,scores,_=recommend_pipeline(uidx,10)
    for model,rec in recs.items():
        p,r,nd,h=calc_metrics(rec,relevant,10)
        results.append([uidx,model,p,r,nd,h])
    if n==1 or n%PROGRESS_EVERY==0 or n==total:
        elapsed=time.time()-t0
        rate=n/elapsed if elapsed else 0
        eta=(total-n)/rate if rate else 0
        print(f"Test: {n}/{total} ({n/total*100:.1f}%)")

results=pd.DataFrame(results,columns=["user","Model","Precision@10","Recall@10","NDCG@10","HitRate@10"])
summary=results.groupby("Model")[["Precision@10","Recall@10","NDCG@10","HitRate@10"]].mean().sort_values("NDCG@10",ascending=False)
display(summary)

Test: 1/4742 (0.0%)
Test: 100/4742 (2.1%)
Test: 200/4742 (4.2%)
Test: 300/4742 (6.3%)
Test: 400/4742 (8.4%)
Test: 500/4742 (10.5%)
Test: 600/4742 (12.7%)
Test: 700/4742 (14.8%)
Test: 800/4742 (16.9%)
Test: 900/4742 (19.0%)
Test: 1000/4742 (21.1%)
Test: 1100/4742 (23.2%)
Test: 1200/4742 (25.3%)
Test: 1300/4742 (27.4%)
Test: 1400/4742 (29.5%)
Test: 1500/4742 (31.6%)
Test: 1600/4742 (33.7%)
Test: 1700/4742 (35.8%)
Test: 1800/4742 (38.0%)
Test: 1900/4742 (40.1%)
Test: 2000/4742 (42.2%)
Test: 2100/4742 (44.3%)
Test: 2200/4742 (46.4%)
Test: 2300/4742 (48.5%)
Test: 2400/4742 (50.6%)
Test: 2500/4742 (52.7%)
Test: 2600/4742 (54.8%)
Test: 2700/4742 (56.9%)
Test: 2800/4742 (59.0%)
Test: 2900/4742 (61.2%)
Test: 3000/4742 (63.3%)
Test: 3100/4742 (65.4%)
Test: 3200/4742 (67.5%)
Test: 3300/4742 (69.6%)
Test: 3400/4742 (71.7%)
Test: 3500/4742 (73.8%)
Test: 3600/4742 (75.9%)
Test: 3700/4742 (78.0%)
Test: 3800/4742 (80.1%)
Test: 3900/4742 (82.2%)
Test: 4000/4742 (84.4%)
Test: 4100/4742 (86.5%)
Test: 420

,Precision@10,Recall@10,NDCG@10,HitRate@10
Model,,,,
Hybrid,0.017777,0.141115,0.078342,0.167229
SVD→ItemKNN,0.016681,0.134317,0.076845,0.157107
SVD,0.016639,0.134410,0.073541,0.156896
ItemKNN,0.011092,0.091557,0.055421,0.106284


## 15. Интерпретация результатов

Полученная таблица позволяет проверить вклад каждого этапа.

Можно сделать вывод о том, что при данной конфигурации гиперпараметров большую часть релевантных ответов вносит SVD. ItemKNN и Контентный подход лишь немного увеличивают эффективность системы, но, тем не менее, дают полезный эффект.

Нельзя не заметить, что все метрики показали довольно низкие результаты. Повторно отмечу, что причина этого в чрезвычайно малых *тестовой* и *валидационной* выборках. На десять рекомендуемых игр приходится 1-3(в большинстве случаев) игр, реально оценённых пользователем, отчего шанс удачного попадания рекомендации очень невысок.

## 16. Итоговый вывод

В работе была построена последовательная гибридная рекомендательная система SVD → ItemKNN → Content-Based. История пользователей разделена примерно на train/validation/test в пропорции 80/10/10.

На первом этапе SVD анализирует матрицу пользователь–игра и формирует широкий пул кандидатов.

На втором этапе ItemKNN уточняет кандидатов на основе похожести игр по пользовательским взаимодействиям.

На третьем этапе Content-Based выполняет финальную персонализацию с использованием непосредственно характеристик игр.

Такой подход объединяет три разных источника информации:
- скрытые предпочтения пользователей;
- поведенческую похожесть игр;
- содержательные характеристики игр.